In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# OBTENEMOS LOS DATOS DE LA CAPA SILVER
gold_silver = spark.table("spotify_catalog.silver.spotify_tracks")

display(gold_silver.limit(10))

In [0]:
date_min = gold_silver.select(min("release_date_clean")).first()[0]
date_max = gold_silver.select(max("release_date_clean")).first()[0]


In [0]:
dim_date = spark.sql(f"""
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT) AS sk_date,
    d AS full_date,
    year(d) AS year,
    quarter(d) AS quarter,
    month(d) AS month,
    date_format(d, 'MMMM') AS month_name,
    weekofyear(d) AS week_of_year,
    dayofmonth(d) AS day_of_month,
    dayofweek(d) AS day_of_week,
    date_format(d, 'EEEE') AS day_name
FROM (
    SELECT explode(
        sequence(
            to_date('{date_min}'),
            to_date('{date_max}'),
            interval 1 day
        )
    ) AS d
)
""")

In [0]:
(
    dim_date
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "spotify_catalog.gold.dim_date"
    )
)